<a href="https://colab.research.google.com/github/BF667/zero-shot-svc/blob/main/zero_shot_svc_colab.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🎤 Zero-Shot Singing Voice Conversion - v2.0 Enhanced

Convert any singing voice to sound like a target speaker — **no training required**.

### ✨ What's New in v2.0
- **Dual-Mode Pipeline**: Signal Processing (instant) + Neural RVC (high quality)
- **Advanced Effects**: Formant shifting, noise reduction, vibrato, breathiness
- **8 Built-in Presets**: Male→Female, Female→Male, Robot, Soft Voice, etc.
- **Batch Processing**: Convert multiple files at once
- **Gradio Demo**: Interactive web UI (new!)

---

| Component | Model | Detail |
|-----------|-------|--------|
| Content Encoder | ContentVec (HuBERT) | 256-dim speaker-invariant |
| F0 Extractor | RMVPE / pyworld | Viterbi-smoothed pitch |
| Speaker Encoder | CAM++ | 192-dim embedding |
| Generator | VITS + FiLM | Conditional mel synthesis |
| Vocoder | HiFi-GAN / Griffin-Lim | 32kHz output |

**🚀 Quick Start**: Run cells 1-3, then either use **Cell 4 (Gradio Demo)** or **Cells 5-7 (CLI-style)**

In [ ]:
#@title ## 1️⃣ Check GPU & Environment

#@markdown Verifies GPU availability and shows system info.

!echo "=== System Info ==="
!cat /etc/os-release | grep PRETTY_NAME
!echo ""
!echo "=== GPU Status ==="
!nvidia-smi --query-gpu=name,memory.total,memory.free,driver_version --format=csv,noheader 2>/dev/null || echo "No GPU detected - will use CPU mode"
!echo ""
!echo "=== Python Version ==="
!python3 --version

In [ ]:
#@title ## 2️⃣ Clone Repository & Install Dependencies

#@markdown Clones the repo and installs all required packages including Gradio for the web UI.

import subprocess
import sys

# Change to content directory
%cd /content

# Clone repository if not exists
import os
if not os.path.exists('zero-shot-svc'):
    print('📥 Cloning repository...')
    !git clone https://github.com/BF667/zero-shot-svc.git
else:
    print('✓ Repository already cloned')

%cd zero-shot-svc

# Install PyTorch with CUDA support
print('\n🔧 Installing PyTorch with CUDA...')
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu121 2>/dev/null || pip install -q torch torchaudio

# Install all dependencies
print('🔧 Installing dependencies...')
!pip install -q -r requirements.txt

# Ensure Gradio is installed for demo
!pip install -q gradio>=4.0

print('\n✅ Installation complete!')

In [ ]:
#@title ## 3️⃣ Download Pretrained Weights (Optional)

#@markdown ---
#@markdown ### Select Conversion Mode:
#@markdown **Signal Processing Mode**: Works immediately without downloads (uses Griffin-Lim vocoder)
#@markdown **Neural Mode**: Requires ~2GB download for high-quality RVC conversion
#@markdown ---

MODE = 'Signal Processing' #@param ['Signal Processing', 'Neural (High Quality)', 'Both']

import sys
sys.path.insert(0, '/content/zero-shot-svc')

if MODE in ['Neural (High Quality)', 'Both']:
    print('📥 Downloading pretrained weights (~2GB)...')
    from weights.download_weights import download_all
    results = download_all(verbose=True)
    print('\n✅ Neural mode ready!')
else:
    print('✅ Signal processing mode ready (no downloads needed)!')
    print('   You can convert audio immediately using Griffin-Lim vocoder.')

In [ ]:
#@title ## 4️⃣ 🎮 Launch Gradio Demo (Interactive Web UI)

#@markdown Launch the full-featured web interface with:
#@markdown - **Convert Tab**: Drag-and-drop audio upload with real-time preview
#@markdown - **Batch Tab**: Process multiple files at once
#@markdown - **History Tab**: View past conversions
#@markdown - **Guide Tab**: Tips and documentation
#@markdown 
#@markdown Features: 8 built-in presets, formant control, noise reduction, vibrato & more!

import sys
sys.path.insert(0, '/content/zero-shot-svc')

# Import Gradio app
from gradio_app import create_demo

# Create and launch the demo
demo = create_demo()

# Launch with public link for easy sharing
demo.launch(
    share=True,           # Creates public link
    server_name='0.0.0.0', # Allow external access
    server_port=7860,     # Standard Gradio port
)

In [ ]:
#@title ## 5️⃣ Upload Audio Files (For CLI-style Conversion)

#@markdown Upload your audio files here if you prefer the programmatic interface over Gradio.
#@markdown 
#@markdown ### Required Files:
#@markdown - **Source audio**: The singing you want to convert (WAV/MP3 recommended)
#@markdown - **Reference audio**: Short clip (5-15s) of the target voice

import os
from google.colab import files

# Create directories
os.makedirs('sample_audio', exist_ok=True)
os.makedirs('sample_audio/output', exist_ok=True)

print('📤 Upload SOURCE audio (the singing to convert):')
uploaded_src = files.upload()
src_name = list(uploaded_src.keys())[0]
os.rename(src_name, f'sample_audio/{src_name}')
SOURCE_PATH = f'sample_audio/{src_name}'

print('\n📤 Upload REFERENCE audio (target voice, 5-15s):')
uploaded_ref = files.upload()
ref_name = list(uploaded_ref.keys())[0]
os.rename(ref_name, f'sample_audio/{ref_name}')
REFERENCE_PATH = f'sample_audio/{ref_name}'

print(f'\n✅ Source:      {SOURCE_PATH} ({os.path.getsize(SOURCE_PATH)/1024:.1f} KB)')
print(f'✅ Reference:   {REFERENCE_PATH} ({os.path.getsize(REFERENCE_PATH)/1024:.1f} KB)')

In [ ]:
#@title ## 6️⃣ Configure & Run Voice Conversion

#@markdown ### Conversion Settings:

#@markdown ---
#@markdown #### Basic Settings
PITCH_SHIFT = 0  #@param {type:"slider", min:-12, max:12, step:1}
#@markdown Semitones (+12 = octave up, -12 = octave down)

#@markdown ---
#@markdown #### Advanced Effects
FORMANT_SHIFT = 0  #@param {type:"slider", min:-6, max:6, step:1}
#@markdown Formant steps (+ = brighter, - = darker, independent of pitch)

NOISE_REDUCTION = 0.0  #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown Background noise removal strength

BREATHINESS = 0.0  #@param {type:"slider", min:0, max:1, step:0.05}
#@markdown Add natural breath sounds (0-1)

#@markdown ---
#@markdown #### Quality Settings
USE_NEURAL = False  #@param {type:"boolean"}
#@markdown Use neural RVC pipeline (requires downloaded weights)

NOISE_SCALE = 0.4  #@param {type:"slider", min:0.1, max:1.0, step:0.05}
#@markdown Generation diversity (neural mode only, lower = more stable)

#@markdown ---
#@markdown #### Presets (overrides manual settings)
PRESET = "Custom" #@param ["Custom", "Male → Female", "Female → Male", "Octave Up", "Octave Down", "Soft/Gentle", "Robot/Synthetic", "Natural", "Helium"]

# Apply preset values
preset_configs = {
    "Custom": {},
    "Male → Female": {"PITCH_SHIFT": 5, "FORMANT_SHIFT": 2, "BREATHINESS": 0.15},
    "Female → Male": {"PITCH_SHIFT": -5, "FORMANT_SHIFT": -2, "BREATHINESS": 0.1},
    "Octave Up": {"PITCH_SHIFT": 12, "FORMANT_SHIFT": 4},
    "Octave Down": {"PITCH_SHIFT": -12, "FORMANT_SHIFT": -4},
    "Soft/Gentle": {"BREATHINESS": 0.25, "NOISE_REDUCTION": 0.2, "NOISE_SCALE": 0.35},
    "Robot/Synthetic": {"NOISE_SCALE": 0.1, "NOISE_REDUCTION": 0.5, "FORMANT_SHIFT": 0},
    "Natural": {"NOISE_REDUCTION": 0.1},
    "Helium": {"PITCH_SHIFT": 8, "FORMANT_SHIFT": 6, "BREATHINESS": 0.2},
}

if PRESET in preset_configs:
    for key, value in preset_configs[PRESET].items():
        locals()[key] = value
        print(f'⚙️ Preset "{PRESET}": {key} = {value}')

print(f'\n🎛️ Configuration:')
print(f'   Pitch Shift:    {PITCH_SHIFT} semitones')
print(f'   Formant Shift:   {FORMANT_SHIFT} steps')
print(f'   Noise Reduction: {NOISE_REDUCTION}')
print(f'   Breathiness:     {BREATHINESS}')
print(f'   Mode:           {"Neural" if USE_NEURAL else "Signal Processing"}')

# Initialize converter
import sys
sys.path.insert(0, '/content/zero-shot-svc')
from pipeline.voice_converter import ZeroShotSVC

device = 'cuda' if USE_NEURAL else 'cpu'
print(f'\n🔄 Initializing on {device.upper()}...')

svc = ZeroShotSVC(device=device, use_neural=USE_NEURAL)

if USE_NEURAL:
    print('📦 Loading neural models...')
    svc.load_models()

# Progress callback
def progress(pct, msg):
    if pct % 10 == 0:  # Print every 10%
        print(f'  [{pct*100:.0f}%] {msg}')

# Run conversion
print('\n🎵 Converting audio...')
output_path = svc.convert(
    source_path=SOURCE_PATH,
    reference_path=REFERENCE_PATH,
    output_path='sample_audio/output/converted.wav',
    f0_transpose=PITCH_SHIFT,
    f0_curve_factor=1.0,
    noise_scale=NOISE_SCALE,
    formant_shift=FORMANT_SHIFT,
    noise_reduction=NOISE_REDUCTION,
    breathiness=BREATHINESS,
    protect_consonants=True,
    progress_callback=progress,
)

print(f'\n✅ Conversion complete! Output saved to: {output_path}')

## 7️⃣ Listen & Download Results

In [ ]:
import IPython.display as ipd
import os

print('🎧 Audio Playback\n')
print('▶️ Source (original singing):')
display(ipd.Audio(SOURCE_PATH))

print('▶️ Reference (target voice):')
display(ipd.Audio(REFERENCE_PATH))

if os.path.exists(output_path):
    print('▶️ Converted output:')
    display(ipd.Audio(output_path))
    print(f'\n📁 File size: {os.path.getsize(output_path)/1024:.1f} KB')
else:
    print('❌ Output file not found')

In [ ]:
from google.colab import files

print('📥 Download converted file:')
files.download(output_path)

In [ ]:
#@title ## 8️⃣ Batch Conversion (Multiple Files)

#@markdown Convert multiple source files at once using the same reference voice.

import os
from google.colab import files
import sys
sys.path.insert(0, '/content/zero-shot-svc')

os.makedirs('sample_audio/batch_output', exist_ok=True)

print('📤 Upload multiple source audio files for batch conversion:')
batch_uploads = files.upload()

# Reuse existing SVC instance or create new one
if 'svc' not in dir():
    from pipeline.voice_converter import ZeroShotSVC
    svc = ZeroShotSVC(device='cpu', use_neural=False)

results = []
for fname in batch_uploads.keys():
    src = f'sample_audio/{fname}'
    os.rename(fname, src)
    out = f'sample_audio/batch_output/{fname}'
    print(f'\n🔄 Converting {fname}...')
    try:
        svc.convert(
            source_path=src,
            reference_path=REFERENCE_PATH,
            output_path=out,
            f0_transpose=PITCH_SHIFT if 'PITCH_SHIFT' in dir() else 0,
            formant_shift=FORMANT_SHIFT if 'FORMANT_SHIFT' in dir() else 0,
        )
        results.append({'file': fname, 'status': '✅ Success'})
        print(f'   Done → {out}')
    except Exception as e:
        results.append({'file': fname, 'status': f'❌ Error: {str(e)}'})
        print(f'   Error: {e}')

print(f'\n{'='*50}')
print(f'BATCH CONVERSION COMPLETE')
print(f'{"="*50}')
for r in results:
    print(f"  {r['status']} {r['file']}")
print(f'\nTotal: {len([r for r in results if "Success" in r["status"]])}/{len(results)} files converted')

# Download all batch outputs
print('\n📥 Downloading all converted files...')
for f in os.listdir('sample_audio/batch_output'):
    if f.endswith('.wav'):
        files.download(f'sample_audio/batch_output/{f}')

In [ ]:
#@title ## 9️⃣ Voice Similarity Analysis

#@markdown Compare how similar two voices are using MFCC and spectral analysis.

import sys
sys.path.insert(0, '/content/zero-shot-svc')
from pipeline.voice_converter import ZeroShotSVC

# Use signal mode for similarity (faster)
svc_sim = ZeroShotSVC(device='cpu', use_neural=False)

print('🔍 Computing voice similarity...')
print(f'   Audio 1: {SOURCE_PATH}')
print(f'   Audio 2: {REFERENCE_PATH}')

similarity = svc_sim.compute_similarity(SOURCE_PATH, REFERENCE_PATH)

print(f'\n{"="*50}')
print('SIMILARITY RESULTS')
print(f'{"="*50}')
print(f'  Overall Similarity:      {similarity["overall_similarity"]:.3f}')
print(f'  MFCC Cosine Similarity:  {similarity["mfcc_cosine_similarity"]:.3f}')
print(f'  Spectral Centroid Corr:   {similarity["spectral_centroid_correlation"]:.3f}')
print(f'  RMS Correlation:          {similarity["rms_correlation"]:.3f}')

# Interpretation
overall = similarity['overall_similarity']
if overall > 0.8:
    interpretation = "🎯 Very similar voices (likely same speaker)"
elif overall > 0.6:
    interpretation = "🔵 Moderately similar"
elif overall > 0.4:
    interpretation = "🟡 Somewhat different"
else:
    interpretation = "🔴 Very different voices"

print(f'\n  Interpretation: {interpretation}')

In [ ]:
#@title ## 🔟 Advanced: Feature Analysis & Visualization

#@markdown Extract and visualize F0 pitch contours, spectral features, and more.

import numpy as np
import matplotlib.pyplot as plt
import sys
sys.path.insert(0, '/content/zero-shot-svc')

# Configure matplotlib for Chinese characters
try:
    import matplotlib.font_manager as fm
    fm.fontManager.addfont('/usr/share/fonts/truetype/chinese/NotoSansSC-Regular.ttf')
    plt.rcParams['font.sans-serif'] = ['Noto Sans SC', 'DejaVu Sans']
except:
    pass
plt.rcParams['axes.unicode_minus'] = False

from pipeline.voice_converter import ZeroShotSVC

# Initialize for feature extraction
svc_feat = ZeroShotSVC(device='cpu', use_neural=False)

print('📊 Extracting features from source audio...')
features = svc_feat.extract_features(SOURCE_PATH)
f0 = features['f0']
voiced_f0 = f0[f0 > 0]

print(f'\n📈 Feature Statistics:')
print(f'   Duration:    {features["duration"]:.1f}s')
print(f'   Content dim: {features["content"].shape}')
print(f'   F0 frames:  {len(f0)}')
if len(voiced_f0) > 0:
    print(f'   F0 range:    {voiced_f0.min():.1f} – {voiced_f0.max():.1f} Hz')
    print(f'   F0 mean:     {voiced_f0.mean():.1f} Hz')
    print(f'   Voiced:      {len(voiced_f0)}/{len(f0)} ({100*len(voiced_f0)/len(f0):.0f}%)')

# Create visualization
fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Plot 1: F0 contour
ax1 = axes[0, 0]
time_axis = np.arange(len(f0)) * 0.02  # 20ms per frame
voiced_mask = f0 > 0
ax1.plot(time_axis[voiced_mask], f0[voiced_mask], 'b-', linewidth=0.8, label='F0')
ax1.fill_between(time_axis, 0, np.where(voiced_mask, f0, 0), alpha=0.3)
ax1.set_xlabel('Time (s)')
ax1.set_ylabel('Frequency (Hz)')
ax1.set_title('F0 Pitch Contour')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot 2: F0 histogram
ax2 = axes[0, 1]
if len(voiced_f0) > 0:
    ax2.hist(voiced_f0, bins=50, color='steelblue', edgecolor='black', alpha=0.7)
    ax2.axvline(voiced_f0.mean(), color='red', linestyle='--', label=f'Mean: {voiced_f0.mean():.0f} Hz')
    ax2.set_xlabel('Frequency (Hz)')
    ax2.set_ylabel('Count')
    ax2.set_title('F0 Distribution')
    ax2.legend()

# Plot 3: Voiced/Unvoiced segments
ax3 = axes[1, 0]
uv_plot = voiced_mask.astype(float)
ax3.plot(time_axis, uv_plot, 'g-', linewidth=0.5)
ax3.fill_between(time_axis, 0, uv_plot, where=uv_plot > 0.5, color='green', alpha=0.3, label='Voiced')
ax3.fill_between(time_axis, 0, uv_plot, where=uv_plot <= 0.5, color='gray', alpha=0.3, label='Unvoiced')
ax3.set_xlabel('Time (s)')
ax3.set_ylabel('Voicing')
ax3.set_title('Voiced/Unvoiced Detection')
ax3.legend()
ax3.set_ylim(-0.1, 1.1)

# Plot 4: F0 range estimation (vocal range)
ax4 = axes[1, 1]
if len(voiced_f0) > 0:
    # Estimate vocal range
    f0_min = voiced_f0.min()
    f0_max = voiced_f0.max()
    
    # Musical note conversion (approximate)
    def freq_to_note(freq):
        if freq <= 0:
            return 'C0'
        notes = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
        note_num = 12 * np.log2(freq / 261.63) + 40  # A4 = 69
        note = notes[int(note_num) % 12]
        octave = int(note_num // 12) - 1
        return f'{note}{octave}'
    
    low_note = freq_to_note(f0_min)
    high_note = freq_to_note(f0_max)
    
    ax4.barh(['Lowest', 'Highest', 'Mean'], [f0_min, f0_max, voiced_f0.mean()], 
            color=['lightcoral', 'lightgreen', 'lightblue'])
    ax4.axvline(261.63, color='gray', linestyle=':', alpha=0.5, label='Middle C')
    ax4.set_xlabel('Frequency (Hz)')
    ax4.set_title(f'Vocal Range: {low_note} - {high_note}')
    ax4.legend()

plt.tight_layout()
plt.show()

# Print summary
if len(voiced_f0) > 0:
    print(f'\n🎤 Estimated Vocal Range: {low_note} to {high_note}')
    print(f'   ({f0_min:.0f} Hz - {f0_max:.0f} Hz)')

In [ ]:
#@title ## 1️⃣1️⃣ Speaker Profile Management

#@markdown Save speaker profiles for reuse and compare embeddings.

import sys
import json
sys.path.insert(0, '/content/zero-shot-svc')
from pipeline.voice_converter import ZeroShotSVC

svc_profile = ZeroShotSVC(device='cpu', use_neural=False)

#@markdown ### Actions
PROFILE_ACTION = "Save Profile" #@param ["Save Profile", "List Profiles", "Compare Embeddings"]

if PROFILE_ACTION == "Save Profile":
    PROFILE_NAME = "My Voice" #@param {type:"string"}
    print(f'💾 Saving speaker profile: {PROFILE_NAME}')
    profile_path = svc_profile.save_speaker_profile(
        reference_path=REFERENCE_PATH,
        name=PROFILE_NAME
    )
    print(f'✅ Saved to: {profile_path}')
    
    # Show profile details
    with open(profile_path) as f:
        profile_data = json.load(f)
    print(f'\nProfile Details:')
    for key, value in profile_data.items():
        if isinstance(value, float):
            print(f'   {key}: {value:.2f}')
        else:
            print(f'   {key}: {value}')

elif PROFILE_ACTION == "List Profiles":
    profiles = svc_profile.list_speaker_profiles()
    print(f'📋 Saved Speaker Profiles ({len(profiles)} total):\n')
    if profiles:
        for p in profiles:
            print(f'  🎤 {p.get("name", "Unknown")}')
            print(f'     Duration: {p.get("duration", 0):.1f}s')
            print(f'     Created:  {p.get("created_at", "Unknown")[:19]}')
            if 'f0_mean' in p:
                print(f'     F0 Mean:  {p["f0_mean"]:.1f} Hz')
            print()
    else:
        print('  No saved profiles found.')

elif PROFILE_ACTION == "Compare Embeddings':
    print('🔬 Computing speaker embeddings...')
    emb_ref = svc_profile.extract_speaker_embedding(REFERENCE_PATH)
    emb_src = svc_profile.extract_speaker_embedding(SOURCE_PATH)
    
    # Cosine similarity
    cos_sim = np.dot(emb_ref, emb_src) / (np.linalg.norm(emb_ref) * np.linalg.norm(emb_src) + 1e-8)
    
    print(f'\nEmbedding Comparison:')
    print(f'   Reference embedding shape: {emb_ref.shape}')
    print(f'   Source embedding shape:    {emb_src.shape}')
    print(f'   Cosine similarity:         {cos_sim:.4f}')
    
    if cos_sim > 0.85:
        print(f'   → Same speaker (very high similarity)')
    elif cos_sim > 0.7:
        print(f'   → Likely same speaker')
    elif cos_sim > 0.5:
        print(f'   → Different speakers but some similarity')
    else:
        print(f'   → Very different speakers')

In [ ]:
#@title ## 1️⃣2️⃣ Start REST API Server (Optional)

#@markdown Launch the FastAPI server for programmatic access.
#@markdown 
#@markdown This is useful for integrating with other applications or services.

import sys
sys.path.insert(0, '/content/zero-shot-svc')

#@markdown ### Server Settings
API_PORT = 8000  #@param {type:"integer"}
PUBLIC_ACCESS = True  #@param {type:"boolean"}

host = '0.0.0.0' if PUBLIC_ACCESS else 'localhost'

print(f'🚀 Starting API Server on http://{host}:{API_PORT}')
print('\nAvailable Endpoints:')
print('  POST   /api/convert       - Convert audio')
print('  POST   /api/batch-convert - Batch convert')
print('  GET    /api/download/{id} - Download result')
print('  POST   /api/similarity    - Voice similarity')
print('  POST   /api/profiles      - Save speaker profile')
print('  GET    /api/profiles      - List profiles')
print('  GET    /api/health         - Health check')
print('  GET    /api/status         - System status')
print('\nAPI Documentation will be available at:')
print(f'  http://localhost:{API_PORT}/docs (Swagger UI)')
print(f'  http://localhost:{API_PORT}/redoc (ReDoc)')

# Start server
import uvicorn
from api_server import app

uvicorn.run(app, host=host, port=API_PORT)

---

## ❓ Troubleshooting & Tips

| Problem | Solution |
|---------|----------|
| **Output sounds robotic** | Lower `noise_scale` to 0.2-0.3; increase `breathiness` slightly |
| **Poor voice similarity** | Use cleaner/dryer reference audio (10-15s, no reverb) |
| **Pitch artifacts/octave errors** | Adjust pitch shift gradually; enable consonant protection |
| **Background noise** | Increase `noise_reduction` to 0.3-0.5 |
| **Too bright/dark sounding** | Adjust `formant_shift`: negative = darker, positive = brighter |
| **CUDA out of memory** | Switch to Signal Processing mode (no GPU needed) |
| **Conversion slow** | Use shorter audio; Signal Processing mode is faster |

### 🔗 Useful Links

- **GitHub**: [BF667/zero-shot-svc](https://github.com/BF667/zero-shot-svc)
- **Resource Guide**: [RESOURCE_GUIDE.md](https://github.com/BF667/zero-shot-svc/blob/main/RESOURCE_GUIDE.md)
- **RVC Community**: [r/RVCAdepts](https://www.reddit.com/r/RVCAdepts/)
- **Model Weights**: [HuggingFace RVC Models](https://huggingface.co/models?other=rvc)

### 📚 References

- [RVC WebUI](https://github.com/RVC-Boss/Retrieval-based-Voice-Conversion-WebUI) — Original architecture
- [RMVPE Paper](https://arxiv.org/abs/2306.15412) — Pitch extraction method
- [VITS Paper](https://arxiv.org/abs/2106.06103) — Generative backbone
- [HiFi-GAN Paper](https://arxiv.org/abs/2010.05646) — Vocoder
- [CAM++ Paper](https://arxiv.org/abs/2210.16711) — Speaker encoder

---

*Made with ❤️ by BF667 | [Zero-Shot SVC v2.0](https://github.com/BF667/zero-shot-svc)*